> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第05单元 — t检验（独立样本与配对样本）
第05单元专属 notebook —— 重现课程中的全部3个t检验示例：具有统计显著性的独立样本检验（教学方法）、不具显著性的独立样本检验（性别）、以及配对样本检验（前测/后测）。

SPSS/PSPP 对应操作：`Analyze > Compare Means > Independent-Samples T Test` / `Paired-Samples T Test`。

In [ ]:
# 先运行这一格（Colab：点击 ▶）。无需安装任何库。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240人班级" —— 第03-08单元统一使用 seed=2026，
    与仓库中的 lop_hoc_240.csv/.sav 完全一致（非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式教学"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式教学")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式教学") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df：", df.shape)

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]

## 1. 独立样本t检验：教学方法 → 成绩提升幅度（`gain`）—— 具有显著性
第一步始终是Levene检验：决定应读取「假设方差相等」还是「不假设方差相等」这一行。

🎯 **这个方法用来回答什么问题？** 独立样本t检验用于检验：两个互不相关的组（不是同一批人）之间观察到的平均差异，是否足够大，足以让我们相信总体中存在真实差异，而不仅仅是随机抽样波动。原假设H0：两组的总体平均值相同。

In [ ]:
g1 = df.loc[df["method"]=="项目式教学", "gain"]
g2 = df.loc[df["method"]=="传统教学", "gain"]
lev_stat, lev_p = stats.levene(g1, g2)
t, p = stats.ttest_ind(g1, g2, equal_var=(lev_p >= 0.05))
print(f"Levene检验：F={lev_stat:.3f}, p={lev_p:.3f} -> {'假设方差相等' if lev_p>=0.05 else '不假设方差相等'}")
print(f"项目式教学：n={len(g1)}, M={g1.mean():.2f}, SD={g1.std(ddof=1):.2f}")
print(f"传统教学：n={len(g2)}, M={g2.mean():.2f}, SD={g2.std(ddof=1):.2f}")
print(f"t({len(g1)+len(g2)-2})={t:.3f}, p={p:.2e}")

#### 📤 实际输出
Levene检验 p=0.535（≥0.05 → 读取「假设相等」一行）。t(238)=7.589, p<0.001。项目式教学 M=8.66 SD=5.77（n=115）；传统教学 M=2.89 SD=6.00（n=125）。✅ 与课程一致。

**APA格式撰写：**「项目式教学组的平均成绩提升（M=8.66, SD=5.77）显著高于（t(238)=7.59, p<0.001）传统教学组（M=2.89, SD=6.00）。\」

## 2. 独立样本t检验：性别 → 数学成绩 —— 不具显著性

In [ ]:
mA = df.loc[df["gender"]=="男", "math"]
mB = df.loc[df["gender"]=="女", "math"]
lev_stat2, lev_p2 = stats.levene(mA, mB)
t2, p2 = stats.ttest_ind(mA, mB, equal_var=(lev_p2 >= 0.05))
print(f"Levene检验：F={lev_stat2:.3f}, p={lev_p2:.3f}")
print(f"男生：n={len(mA)}, M={mA.mean():.2f}, SD={mA.std(ddof=1):.2f}")
print(f"女生：n={len(mB)}, M={mB.mean():.2f}, SD={mB.std(ddof=1):.2f}")
print(f"t({len(mA)+len(mB)-2})={t2:.3f}, p={p2:.3f}")

#### 📤 实际输出
Levene检验 p=0.081（≥0.05但接近阈值，应谨慎对待）。t(238)=-1.348, p=0.179 → **证据不足**，无法证明该样本中存在性别差异。「未发现差异」的结果同样有价值，应如实报告，而不应「数据挖掘」到出现显著性为止。

## 3. 配对样本t检验：前测 → 后测（同一240名学生）
配对检验比独立检验更有效力，因为它消除了学生个体间的变异 —— 只留下由学习效果导致的变异。

🎯 **这个方法用来回答什么问题？** 配对样本t检验用于检验：当同一组研究对象被测量两次（前测/后测，或两个不同变量）时，两次测量的平均差值是否足够大，足以认为这是一种真实的变化（而非偶然）。原假设H0：平均差值=0。

In [ ]:
t3, p3 = stats.ttest_rel(df["posttest"], df["pretest"])
diff = df["posttest"] - df["pretest"]
print(f"前测平均值={df['pretest'].mean():.2f}（SD={df['pretest'].std(ddof=1):.2f}）")
print(f"后测平均值={df['posttest'].mean():.2f}（SD={df['posttest'].std(ddof=1):.2f}）")
print(f"平均差值={diff.mean():.2f}")
print(f"t({len(df)-1})={t3:.2f}, p={p3:.2e}")
r_pp = stats.pearsonr(df["pretest"], df["posttest"])[0]
print(f"前测-后测相关系数 r={r_pp:.3f}（解释了为何配对检验比独立检验更有效力）")

#### 📤 实际输出
平均差值=5.65，t(239)=13.37, p<0.001。前测-后测相关系数 r≈0.857 —— 非常高，正是配对检验在相同样本量下效力远高于独立检验的原因。✅ 与课程一致。